In [1]:
%pip install numpy pandas scipy matplotlib scikit-learn
import numpy as np
import os
import pandas as pd
from scipy.io import loadmat 
from pathlib import Path
import matplotlib.pyplot as plt
#DATA_DIR = Path(".")        # "." = this folder, where the notebook is

#mat = loadmat(DATA_DIR / "B0005.mat")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
batteries = ["B0005", "B0006", "B0007", "B0018"]
first_capacity = {}                              # an empty dictionary to fill

for name in batteries:
    mat = loadmat(f"data/{name}.mat")
    raw = mat[name][0, 0]["cycle"]
    for i in range(raw.shape[1]):
        c = raw[0, i]
        if c["type"][0] != "discharge":
            continue
        cap = c["data"][0, 0]["Capacity"].flatten()
        if cap.size == 0:
            continue
        first_capacity[name] = float(cap[0])     # store it under the battery's name
        break              
plt.figure()
rows=[]
for name in batteries:                       # do the same work for each cell
    mat = loadmat(f"data/{name}.mat")        # f"..{name}.." puts the name into the path
    raw = mat[name][0, 0]["cycle"]
    baseline=first_capacity[name]
    cycles = []
    k=0
    capacity = []
    first_cycle_below = None 
    n = 0
    starting_voltage_for_battery = None
    for i in range(raw.shape[1]):
        c = raw[0, i]
        if c["type"][0] != "discharge":
            continue
        cap = c["data"][0, 0]["Capacity"].flatten()
        if cap.size == 0:
            continue
        n += 1
        cycles.append(n)
        capacity.append(float(cap[0]))
        state_of_health=(np.array(capacity)/baseline)*100
        last_capacity=capacity[-1]
    for o in capacity:
        k+=1
        if o<=1.4:
            first_cycle_below=k
            break
          
    fade = 100 - (last_capacity / baseline * 100)

    rows.append({
        "battery": name, 
        "cycles": n, 
        "first capacity": baseline,
        "last capacity": last_capacity,
        "first cycle below 1,4Ah": first_cycle_below,
        "fade%":fade
    })
    

    #plt.plot(cycles, capacity, marker="o", markersize=3, label=name)   # one line per cell
df=pd.DataFrame(rows)



#plt.axhline(y=1.40, color="red", linestyle="--", label="Threshold Limit")
#plt.xlabel("Discharge cycle")
#plt.ylabel("Capacity (Ah)")

#plt.title("Capacity degradation")

#plt.grid(True)




<Figure size 640x480 with 0 Axes>

In [3]:
batteries = ["B0005", "B0006", "B0007", "B0018"]
first_capacity = {}                              # an empty dictionary to fill

for name in batteries:
    mat = loadmat(f"data/{name}.mat")
    raw = mat[name][0, 0]["cycle"]
    for i in range(raw.shape[1]):
        c = raw[0, i]
        if c["type"][0] != "discharge":
            continue
        cap = c["data"][0, 0]["Capacity"].flatten()
        if cap.size == 0:
            continue
        first_capacity[name] = float(cap[0])     # store it under the battery's name
        break              
rows=[]
all_trimmed_cycles = []
cycle_features = []
for name in batteries:                       # do the same work for each cell
    mat = loadmat(f"data/{name}.mat")        # f"..{name}.." puts the name into the path
    raw = mat[name][0, 0]["cycle"]
    baseline=first_capacity[name]
    cycles = []
    k=0
    capacity = []
    first_cycle_below = None 
    n = 0
    for i in range(raw.shape[1]):
        c = raw[0, i]
        if c["type"][0] != "discharge":
            continue
        cap = c["data"][0, 0]["Capacity"].flatten()
        if cap.size == 0:
            continue
        # Get the measurements for this one discharge cycle
        data = c["data"][0, 0]

        # Put the measurements into a small table
        cycle_df = pd.DataFrame({
            "time": np.asarray(data["Time"]).flatten(),
            "current": np.asarray(data["Current_measured"]).flatten(),
            "voltage": np.asarray(data["Voltage_measured"]).flatten(),
            "temperature": np.asarray(data["Temperature_measured"]).flatten()

        })

        
        is_discharging = cycle_df["current"].abs() > 0.05
        if not is_discharging.any():
            print(f"Skipping {name}, cycle position {i}: no discharge samples found")
            continue
        active_times = cycle_df.loc[is_discharging, "time"]
        first_discharge_time = active_times.iloc[0]
        
        cycle_df["seconds_since_start"] = (
        cycle_df["time"] - first_discharge_time
            )
        
        keep = is_discharging & (cycle_df["seconds_since_start"] <= 1500)
        trimmed_cycle = cycle_df.loc[keep].copy()

        trimmed_cycle["battery"] = name
        trimmed_cycle["cycle"] = n + 1
       # Starting and ending voltage in the 1500-second window
        starting_voltage = trimmed_cycle["voltage"].iloc[0]
        if n == 0:
            starting_voltage_for_battery = starting_voltage
        end_voltage = trimmed_cycle["voltage"].iloc[-1]
        voltage_drop = starting_voltage - end_voltage
        mean_voltage = trimmed_cycle["voltage"].mean()
        slope_of_voltage_time, voltage_intercept = np.polyfit(
            trimmed_cycle["seconds_since_start"],
            trimmed_cycle["voltage"],
            deg=1
        )

        # Time when voltage first reaches each threshold
        below_3_9 = trimmed_cycle[trimmed_cycle["voltage"] <= 3.9]
        if len(below_3_9) > 0:
            time_to_3_9 = below_3_9["seconds_since_start"].iloc[0]
        else:
          time_to_3_9 = np.nan  # it did not reach 3.9 V in this window

        below_3_8 = trimmed_cycle[trimmed_cycle["voltage"] <= 3.8]
        if len(below_3_8) > 0:
         time_to_3_8 = below_3_8["seconds_since_start"].iloc[0]
        else:
                time_to_3_8 = np.nan

        below_3_7 = trimmed_cycle[trimmed_cycle["voltage"] <= 3.7]
        if len(below_3_7) > 0:
          time_to_3_7 = below_3_7["seconds_since_start"].iloc[0]
        else:
            time_to_3_7 = np.nan

        # Temperature at the start and end of the 1500-second window.
        starting_temperature = trimmed_cycle["temperature"].iloc[0]
        ending_temperature = trimmed_cycle["temperature"].iloc[-1]
        temperature_rise = ending_temperature - starting_temperature
        maximum_temperature = trimmed_cycle["temperature"].max()

        # Fit a straight line to estimate temperature change over time.
        temperature_slope_per_second, temperature_intercept = np.polyfit(
            trimmed_cycle["seconds_since_start"],
            trimmed_cycle["temperature"],
            deg=1
        )
        temperature_slope_per_minute = temperature_slope_per_second * 60

        # Voltage times current is power; integrate it to get energy.
        power_watts = trimmed_cycle["voltage"] * trimmed_cycle["current"].abs()
        energy_wh = np.trapezoid(
            power_watts,
            x=trimmed_cycle["seconds_since_start"]
        ) / 3600
        mean_discharge_current = trimmed_cycle["current"].abs().mean()

        # Use the full cycle, not only the 1500-second window.
        full_discharge_duration = active_times.iloc[-1] - active_times.iloc[0]

        # Save one summary row for this discharge cycle.
        cycle_features.append({
            "battery": name,
            "cycle": n + 1,
            "starting voltage (V)": starting_voltage,
            "ending voltage in window (V)": end_voltage,
            "voltage drop in window (V)": voltage_drop,
            "mean voltage in window (V)": mean_voltage,
            "voltage slope (V/s)": slope_of_voltage_time,
            "time to 3.9 V (s)": time_to_3_9,
            "time to 3.8 V (s)": time_to_3_8,
            "time to 3.7 V (s)": time_to_3_7,
            "starting temperature (C)": starting_temperature,
            "ending temperature (C)": ending_temperature,
            "temperature rise (C)": temperature_rise,
            "temperature slope (C/min)": temperature_slope_per_minute,
            "maximum temperature (C)": maximum_temperature,
            "energy in window (Wh)": energy_wh,
            "mean discharge current (A)": mean_discharge_current,
            "full discharge duration (s)": full_discharge_duration,
            "capacity (Ah)": float(cap[0]),
            "SOH (%)": float(cap[0]) / baseline * 100,
        })
        trimmed_cycle["mean_voltage"] = mean_voltage
        trimmed_cycle["voltage_drop"] = voltage_drop
        trimmed_cycle["voltage_slope"] = slope_of_voltage_time

        all_trimmed_cycles.append(trimmed_cycle)

        n += 1
        cycles.append(n)
        capacity.append(float(cap[0]))
        state_of_health=(np.array(capacity)/baseline)*100        
        last_capacity=capacity[-1]
    for o in capacity:
        k+=1
        if o<=1.4:
            first_cycle_below=k
            break
          
    fade = 100 - (last_capacity / baseline * 100)
    
    rows.append({
        "battery": name, 
        "cycles": n, 
        "first capacity": baseline,
        "last capacity": last_capacity,
        "first cycle below 1,4Ah": first_cycle_below,
        "fade%":fade,
        "starting voltage (V)": starting_voltage_for_battery,

    })
    
cycle_features_df=pd.DataFrame(cycle_features)
    
features_to_compare = [
    "full discharge duration (s)",
    "voltage slope (V/s)",
    "time to 3.7 V (s)",
    "voltage drop in window (V)",
    "ending voltage in window (V)",
    "time to 3.8 V (s)",
    "mean voltage in window (V)",
    "temperature rise (C)",
    "temperature slope (C/min)",
    "energy in window (Wh)",
]
correlation_rows=[]
for feature in features_to_compare:
    correlation_value=cycle_features_df[feature].corr(
        cycle_features_df["capacity (Ah)"]
    )
    correlation_rows.append({
        "feature":feature,
        "correlation with capacity": correlation_value
    })

correlation_table = pd.DataFrame(correlation_rows).round(4)
display(correlation_table)

df=pd.DataFrame(rows)

all_trimmed_cycles_df = pd.concat(all_trimmed_cycles, ignore_index=True)

cycle_features_df = pd.DataFrame(cycle_features)
#cycle_features_df.head()

#print(all_trimmed_cycles_df)

,feature,correlation with capacity
0,full discharge duration (s),0.9974
1,voltage slope (V/s),0.9820
2,time to 3.7 V (s),0.9573
3,voltage drop in window (V),-0.9707
4,ending voltage in window (V),0.9288
5,time to 3.8 V (s),0.9167
6,mean voltage in window (V),0.9099
7,temperature rise (C),-0.8983
8,temperature slope (C/min),-0.8583
9,energy in window (Wh),0.8467


In [4]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


partial_features = [
    "voltage slope (V/s)",
    "time to 3.7 V (s)",
    "voltage drop in window (V)",
    "temperature rise (C)",
    "energy in window (Wh)",
    "mean discharge current (A)"
]

X = cycle_features_df[partial_features]
y = cycle_features_df["SOH (%)"]
groups = cycle_features_df["battery"] 

logo = LeaveOneGroupOut()
maes = []
r2s = []

for fold_idx, (train_index, test_index) in enumerate(logo.split(X, y, groups)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    test_battery_name = groups.iloc[test_index].values[0]
    
    model = make_pipeline(StandardScaler(), LinearRegression())
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_test)
    
    fold_mae = mean_absolute_error(y_test, predictions)
    fold_r2 = r2_score(y_test, predictions)
    
    maes.append(fold_mae)
    r2s.append(fold_r2)
    
    print(f"--- Test Battery: {test_battery_name} ---")
    print(f"  MAE: {fold_mae:.4f}")
    print(f"  R² Score: {fold_r2:.4f}\n")

print("==============================")
print(f"Average MAE across all cells: {np.mean(maes):.4f}")
print(f"Average R² across all cells: {np.mean(r2s):.4f}")
full_features = partial_features + ["full discharge duration (s)"]

ImportError: DLL load failed while importing _loss: An Application Control policy has blocked this file.